# Phase 4 — Symbolic Music Generation

This notebook demonstrates how to export chord progressions — both synthetic and extracted from real Lakh songs — to **MIDI** and **MusicXML**.

**Workflow:**
1. Static demo progression → `.mid` and `.musicxml`
2. Load a real song row from the curated dataset → re-export its top chords
3. Round-trip validation: parse the generated MIDI back with `MIDIHarmonicAnalyzer` and compare features

**Run from the `KG_DL_PROJECT/` root directory.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import music21 as m21
import matplotlib.pyplot as plt

from harmonic_kg_project.config.config              import CFG
from harmonic_kg_project.generation                 import write_chord_midi, write_musicxml
from harmonic_kg_project.dataset.harmonic_analyzer  import MIDIHarmonicAnalyzer

OUT_DIR = os.path.join(os.path.dirname(''), '../../04_generation/output')
os.makedirs(OUT_DIR, exist_ok=True)

DEMO_MIDI  = os.path.join(OUT_DIR, 'demo.mid')
DEMO_XML   = os.path.join(OUT_DIR, 'demo.musicxml')
SONG_MIDI  = os.path.join(OUT_DIR, 'song_from_dataset.mid')
SONG_XML   = os.path.join(OUT_DIR, 'song_from_dataset.musicxml')

print('Output dir:', os.path.abspath(OUT_DIR))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  Static demo: I–VI–IV–V progression in C major
# ─────────────────────────────────────────────────────────────────────────────
gen_cfg = CFG['generation']

demo_chords = ["C:maj", "A:min", "F:maj", "G:7"]

write_chord_midi(
    chords            = demo_chords,
    output_path       = DEMO_MIDI,
    key               = gen_cfg['default_key'],
    tempo             = gen_cfg['default_tempo'],
    duration_per_chord= gen_cfg['duration_per_chord'],
)
print('Written:', DEMO_MIDI)

write_musicxml(
    chords                 = demo_chords,
    output_path            = DEMO_XML,
    key                    = gen_cfg['default_key'],
    tempo                  = gen_cfg['default_tempo'],
    duration_per_chord     = gen_cfg['duration_per_chord'],
    add_roman_annotations  = True,
    verbose                = True,
)
print('Written:', DEMO_XML)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  Parse generated MIDI back with music21 for a quick sanity check
# ─────────────────────────────────────────────────────────────────────────────
score = m21.converter.parse(DEMO_MIDI)
score.show('text')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  Round-trip: analyse the generated MIDI with MIDIHarmonicAnalyzer
# ─────────────────────────────────────────────────────────────────────────────
analyzer  = MIDIHarmonicAnalyzer()
rpt_demo  = analyzer.analyze(DEMO_MIDI, verbose=True)
feats     = analyzer.extract_features(rpt_demo)
analyzer.print_features(feats)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  Generate from a real song in the curated dataset
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_parquet(CFG['dataset']['parquet_file'])
print(f'Dataset loaded: {len(df):,} songs')

# Pick a representative song (e.g., first row with a known genre)
sample = df.dropna(subset=['primary_genre']).iloc[0]
print(f'\nSong: {sample.get("title","(no title)")} by {sample.get("artist_name","(unknown)")}')
print(f'Genre: {sample["primary_genre"]} | Key: {sample.get("global_key","?")}')

# 'top_chords' is a list of (harte_label, count) tuples serialised as string
# We recover the Harte labels in order of frequency.
import ast
raw_top = sample.get('top_chords', '[]')
top_chords_parsed = ast.literal_eval(raw_top) if isinstance(raw_top, str) else raw_top

# Flatten: [(label, count), ...] → [label, ...] ordered by count desc
song_chords = [lbl for lbl, _cnt in sorted(top_chords_parsed, key=lambda x: -x[1])]
print(f'Top {len(song_chords)} chords: {song_chords}')

In [ ]:
if song_chords:
    key_str = sample.get('global_key', gen_cfg['default_key'])

    write_chord_midi(
        chords             = song_chords,
        output_path        = SONG_MIDI,
        key                = key_str,
        tempo              = sample.get('msd_tempo', gen_cfg['default_tempo']),
        duration_per_chord = gen_cfg['duration_per_chord'],
    )
    print('Written MIDI:', SONG_MIDI)

    write_musicxml(
        chords                = song_chords,
        output_path           = SONG_XML,
        key                   = key_str,
        tempo                 = sample.get('msd_tempo', gen_cfg['default_tempo']),
        duration_per_chord    = gen_cfg['duration_per_chord'],
        add_roman_annotations = True,
    )
    print('Written MusicXML:', SONG_XML)
else:
    print('No chords available for this song; skipping generation.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  Compare original vs. re-exported using harmonic features
# ─────────────────────────────────────────────────────────────────────────────
if song_chords and os.path.exists(SONG_MIDI):
    rpt_song  = analyzer.analyze(SONG_MIDI, verbose=False)
    feats_gen = analyzer.extract_features(rpt_song)

    # Pick a handful of features to compare side-by-side
    compare_keys = ['num_unique_chords', 'transition_entropy', 'func_ratio_T',
                    'func_ratio_D', 'func_ratio_S', 'ic1', 'ic4', 'ic5']

    orig_vals = {k: sample.get(k, float('nan')) for k in compare_keys}
    gen_vals  = {k: feats_gen.get(k, float('nan')) for k in compare_keys}

    cmp_df = pd.DataFrame({'original': orig_vals, 'generated': gen_vals})
    print(cmp_df.round(4))

    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(compare_keys))
    ax.bar([i - 0.2 for i in x], cmp_df['original'], width=0.4, label='original')
    ax.bar([i + 0.2 for i in x], cmp_df['generated'], width=0.4, label='generated')
    ax.set_xticks(list(x)); ax.set_xticklabels(compare_keys, rotation=30, ha='right')
    ax.set_title('Original vs. Generated — Selected Harmonic Features')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('Skipping comparison (no song MIDI).')